# Задание 2

Для каждого `user_id` из журнала визитов нужно найти категорию покупки и записать в `funnel.csv` только те визиты, для которых покупка существует. Результат должен содержать столбцы `user_id`, `source`, `category`.

В условии упомянут `visit_log.csv`, но приложен файл `visit_log.xlsx`. Поэтому ниже Excel-файл читается в потоковом режиме: `read_only=True` не загружает весь лист в оперативную память. Словарь покупок, напротив, целиком хранится в памяти, как и требуется в условии.


In [1]:
import csv
import json
from itertools import islice
from pathlib import Path

from openpyxl import load_workbook


Сначала используем решение первого задания: читаем `purchase_log.txt` построчно и строим словарь вида `{user_id: category}`.


In [2]:
def load_purchases(file_path: str | Path) -> dict[str, str]:
    purchases: dict[str, str] = {}

    with open(file_path, encoding="utf-8") as file:
        next(file)  # пропускаем строку-заголовок
        for line in file:
            purchase = json.loads(line)
            purchases[purchase["user_id"]] = purchase["category"]

    return purchases


Затем обходим строки Excel-файла по одной. Поиск категории в словаре выполняется через `purchases.get(user_id)`. Если категории нет, визит пропускается; если есть — записывается в `funnel.csv`. Параметр `newline=""` нужен модулю `csv`, чтобы не создавать пустые строки между записями.


In [3]:
def create_funnel(
    purchases: dict[str, str],
    visit_path: str | Path,
    output_path: str | Path,
) -> int:
    """Записывает визиты с покупкой и возвращает их количество."""
    workbook = load_workbook(visit_path, read_only=True, data_only=True)
    worksheet = workbook.active

    try:
        rows = worksheet.iter_rows(values_only=True)
        headers = next(rows)
        user_id_index = headers.index("user_id")
        source_index = headers.index("source")

        purchase_count = 0
        with open(output_path, "w", encoding="utf-8", newline="") as output_file:
            writer = csv.writer(output_file)
            writer.writerow(["user_id", "source", "category"])

            for row in rows:
                user_id = row[user_id_index]
                category = purchases.get(user_id)
                if category is not None:
                    writer.writerow([user_id, row[source_index], category])
                    purchase_count += 1

        return purchase_count
    finally:
        workbook.close()


In [4]:
purchases = load_purchases("purchase_log.txt")
purchase_count = create_funnel(purchases, "visit_log.xlsx", "funnel.csv")

print(f"Записано строк с покупками: {purchase_count}")


Записано строк с покупками: 13638


Проверим заголовок и первые две записи созданного файла (то есть первые три строки вместе с заголовком).


In [5]:
with open("funnel.csv", encoding="utf-8") as file:
    for line in islice(file, 3):
        print(line.rstrip())


user_id,source,category
1840e0b9d4,other,Продукты
4e4f90fcfb,context,Электроника
